## VisionDES: Example Usage 

In [ ]:
pip install vision-des==0.0.2

In [2]:
import os
import timm 
import faiss
import torch 
from torch.utils.data import DataLoader
from torchvision import datasets, transforms 

from torchvision.models import (resnet101, vgg19, googlenet, efficientnet_b0, mobilenet_v2, inception_v3) 
from torchvision.models.vision_transformer import vit_b_16 

from vision_des import VisionDES

import warnings
warnings.filterwarnings("ignore")

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda:0


### Define dataset 

In [4]:
print(f"Loading dataset with resize transform...")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]) 
])

val_data_dir = 'imagenet_validation' 
test_data_dir = 'test1'

val_dataset = datasets.ImageFolder(os.path.join(val_data_dir), transform=transform) 
test_dataset = datasets.ImageFolder(os.path.join(test_data_dir), transform=transform) 

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4) 
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)  

print(f"Validation samples: {len(val_dataset)}")
print(f"Validation samples: {len(test_dataset)}")

Loading dataset with resize transform...
Validation samples: 30000
Validation samples: 2002


In [8]:
trained_pool = [
    vit_b_16(pretrained=True).to(device).eval(),
    resnet101(pretrained=True).to(device).eval(),
    # efficientnet_b0(pretrained=True).to(device).eval(),
    # vgg19(pretrained=True).to(device).eval(),
    # mobilenet_v2(pretrained=True).to(device).eval(),
    # inception_v3(pretrained=True).to(device).eval(), 
    # NormalizedModel(googlenet(pretrained=True, aux_logits=True, transform_input=False), mean, std).to(device).eval()
]

### VisionDES 

In [9]:
des_model = VisionDES(val_dataset, trained_pool, device)
des_model.fit()

Extracting embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 938/938 [04:05<00:00,  3.82it/s]


In [ ]:
test_img, label = test_dataset[37] 

des_model.predict(test_img, k=7, explain=True, use_fire=False) 